In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter
from plot_utils import save_and_trim

path = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
sys.path.append(path)
print(path)

from utility import*

In [ ]:

def plot_fct_results(fct_results_df, variable_name, variable_values, labels, fct_metric, fig_name):
    """
    Plot FCT results with one subplot for each network, using logarithmic scales and custom ticks,
    ensuring the entire plot has a fixed width, and display a single legend.

    Args:
        fct_results_df (dict): Dictionary of DataFrames containing FCT data for each network.
        variable_name (str): Name of the variable being varied (e.g., "load" or "seed").
        variable_values (list): List of variable values.

    Returns:
        None
    """
    num_networks = len(fct_results_df)-1
    total_width = LINE_WIDTH # Fixed total width in inches
    # height = 0.3 * total_width  # Fixed height in inches
    height = 0.4 * total_width  # Fixed height in inches

    fig, axes = plt.subplots(1, num_networks, figsize=(total_width, height), sharey=False, dpi=DEFAULT_DPI)

    if num_networks == 1:
        axes = [axes]  # Ensure axes is iterable when there is only one subplot

    # Define custom tick patterns
    xticks = XTICKS_HD
    first_yticks = YTICKS_HD_AVG if fct_metric=="avg" else (YTICKS_HD_P99 if fct_metric=="p99" else [])
    # other_yticks = [0.6, 0.8, 1, 1.2, 1.4, 1.6, 1.8, 2, 2.5, 3.0, 5.0]
    other_yticks = [0.5, 1, 1.5, 2, 2.5, 3, 3.5]

    all_handles = []  # To collect all legend handles
    all_labels = []   # To collect all legend labels

    for idx, (network_name, df) in enumerate(fct_results_df.items()):
        if idx == 0:
            continue
        ax = axes[idx-1]

        # Shade background regions
        small_flows_label = f"Small flows"
        large_flows_label = f"Large flows"
        cutoff_label = f"Cut off\n(60 MB)" # Draw a vertical line at x = 60,000,000

        small_flows_patch = ax.axvspan(xmin=0, xmax=CUTOFF, facecolor="lightblue", alpha=DEFAULT_ALPHA, label=small_flows_label)
        large_flows_patch = ax.axvspan(xmin=CUTOFF, xmax=max(xticks), facecolor="lightgreen", alpha=DEFAULT_ALPHA, label=large_flows_label)
        vline = ax.axvline(x=CUTOFF, color="grey", linestyle="--", linewidth=0.75, label=cutoff_label)

        # Ensure these are added to the legend only once
        if small_flows_label not in all_labels:
            all_handles.append(small_flows_patch)
            all_labels.append(small_flows_label)
        if large_flows_label not in all_labels:
            all_handles.append(large_flows_patch)
            all_labels.append(large_flows_label)
        if cutoff_label not in all_labels:
            all_handles.append(vline)
            all_labels.append(cutoff_label)


        for variable in variable_values:
            column_name = f"{variable_name.capitalize()} {variable}"
            if column_name in df:
                # Plot and collect handles and labels
                line, = ax.plot(df["Size"], df[column_name], marker="o", markersize=1.5, linewidth=0.5, label=f"{variable_name.capitalize()} {round(float(variable))}%")
                # line, = ax.plot(df["Size"], df[column_name], marker="o", markersize=1.5, linewidth=0.5, label=f"$L'$ = {float(variable)/10}")
                if line.get_label() not in all_labels:  # Avoid duplicate labels
                    all_handles.append(line)
                    all_labels.append(line.get_label())

        

        # Set logarithmic scale
        if idx == 0:
            ax.set_yscale("log")
        ax.set_xscale("log")
        ax.set_xlim(xticks[0], xticks[-1])
        ax.set_xticks(xticks)
        ax.set_title(f"{labels[idx]}", fontsize=FONT_SIZE-1)


        # Customize Y-axis
        y_ticks = first_yticks if idx == 0 else other_yticks
        ax.set_yticks(y_ticks)
        ax.set_ylim(y_ticks[0], y_ticks[-1])
        if idx != 0:
            ax.yaxis.set_major_formatter(StrMethodFormatter("{x}"))

        

        # Add labels and grid
        ax.set_xlabel("Flow size (bytes)", fontsize=FONT_SIZE-2)
        ax.set_ylabel(f"Norm. {fct_metric} FCT" if idx == 1  else "", fontsize=FONT_SIZE-2, labelpad=5)
        ax.tick_params(axis='both', labelsize=FONT_SIZE-2, length=2.5, width=0.5)
        ax.grid(True, linewidth=0.25)

        # Set subplot frame (spines) linewidth
        for spine in ax.spines.values():
            spine.set_linewidth(0.5)

    # Add a single legend
    fig.legend(
        handlelength=1.25,
        markerscale=1.0,
        handles=all_handles,
        labels=all_labels,
        loc="center right",  # Position legend at the right middle
        fontsize=FONT_SIZE-3,
        frameon=True
    )

    # Adjust layout to make space for the legend and reduce subplot spacing
    
    plt.tight_layout(rect=[0, 0, 0.92, 1])  # Adjust layout to make space for the legend
    
    w = -0.01
    for idx, ax in enumerate(axes):
    # Adjust the position of the subplot
        pos = ax.get_position()  # Get current position
        
        if idx == 0:  # Example adjustment for the second subplot
            ax.set_position([pos.x0, pos.y0, pos.width + w, pos.height])  # Shift subplot to the right
        elif idx == 1:
            ax.set_position([pos.x0 - 0.05, pos.y0, pos.width + w, pos.height])
        elif idx == 2:
            ax.set_position([pos.x0 - 0.2, pos.y0 , pos.width + w, pos.height])

    save_and_trim(f"{FIG_DIR}/{fig_name}_fixed.png", dpi=DEFAULT_DPI)
    # Save the figure with equal cropping at the top and bottom
    plt.show()




In [ ]:
# Example usage
exp_name = f"FCT_microsoft_MegaSwitch"
file_template = "../../results/FCT_microsoft/log_{network}_HD_{variable}pload_seed={seed}.txt"
# networks = ["cbb_108i","opera_ecmp","megaswitch_prio_tor=108_hpr=6_b=3_interval=0.01_reconfigTime=0.01",  "megaswitch_prio_tor=108_hpr=6_b=3_interval=0.01_reconfigTime=0.001", "megaswitch_prio_tor=108_hpr=6_b=4_interval=0.01_reconfigTime=0.01"]
networks = ["cbb_108i","megaswitch_prio_tor=108_hpr=6_b=3_interval=0.01_reconfigTime=0.01", "megaswitch_prio_tor=108_hpr=6_b=4_interval=0.01_reconfigTime=0.01"]
labels = ["CBB-Net", "MegaSwitch (b=3)", "MegaSwitch (b=4)"]
fct_metric = "avg"  # Choose between "avg" and "p99"
variable_name = "load"  # Can be "load" or "seed"
variable_values = ["2.00", "4.00", "6.00"] # List of variable values (e.g., loads or seeds)

# Compute and get DataFrames
seeds = ["1", "2", "3", "4", "5"]

fig_name = f"{fct_metric}_{exp_name}_HD"
fct_results_df = compute_fct_data_average(file_template, networks, fct_metric, variable_name, variable_values, seeds)

# display(fct_results_df['opera_ecmp'])
plot_fct_results(fct_results_df, variable_name, variable_values, labels, fct_metric, fig_name)
manually_crop_figure(f"{FIG_DIR}/{fig_name}_fixed.png", f"{FIG_DIR}/{fig_name}_cropped_fixed.png", crop_left=0, crop_top=50, crop_right=0, crop_bottom=50)

In [ ]:
fct_metric = "p99"  # Choose between "avg" and "p99"

# Compute and get DataFrames
fig_name = f"{fct_metric}_{exp_name}_HD"
fct_results_df = compute_fct_data_average(file_template, networks, fct_metric, variable_name, variable_values, seeds)
# display(fct_results_df['opera_ecmp'])
plot_fct_results(fct_results_df, variable_name, variable_values, labels, fct_metric, fig_name)